# LoanXAI — Model Training & SHAP Analysis Notebook
**Dataset:** Home Credit Default Risk (Kaggle)  
**Model:** XGBoost Classifier  
**XAI:** SHAP TreeExplainer  
**Final Year Project**

---
Run this notebook to:
1. Load and explore the dataset
2. Engineer features
3. Train the XGBoost model
4. Evaluate with AUC-ROC
5. Generate SHAP explanations
6. Save `model.pkl` for the Flask backend

## Step 1: Install Dependencies

In [ ]:
# Run this cell once
!pip install xgboost shap imbalanced-learn pandas numpy scikit-learn matplotlib seaborn

## Step 2: Load Dataset

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Download from: https://www.kaggle.com/competitions/home-credit-default-risk
# Place application_train.csv in the data/ folder

df = pd.read_csv('../data/application_train.csv')
print('Shape:', df.shape)
print('Default rate:', df['TARGET'].mean().round(3))
df.head(3)

## Step 3: Explore & Clean

In [ ]:
# Class distribution
print('Class distribution:')
print(df['TARGET'].value_counts())
print(f'\nDefault rate: {df["TARGET"].mean()*100:.1f}%  (class imbalance!)')

# Missing values
missing = df.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)
print(f'\nColumns with missing values: {len(missing)}')
print(missing.head(20))

## Step 4: Feature Engineering

In [ ]:
# ── One-hot encode categoricals ──
categorical_cols = df.select_dtypes(include='object').columns.tolist()
print('Categorical columns:', len(categorical_cols))

df_encoded = pd.get_dummies(df, columns=categorical_cols, drop_first=False)

# ── Create ratio features ──
df_encoded['INCOME_CREDIT_RATIO']  = df_encoded['AMT_INCOME_TOTAL'] / df_encoded['AMT_CREDIT'].replace(0, np.nan)
df_encoded['ANNUITY_INCOME_RATIO'] = df_encoded['AMT_ANNUITY']      / df_encoded['AMT_INCOME_TOTAL'].replace(0, np.nan)
df_encoded['AGE_YEARS']            = np.abs(df_encoded['DAYS_BIRTH'])   / 365
df_encoded['EMPLOYED_YEARS']       = np.abs(df_encoded['DAYS_EMPLOYED'])/ 365

# Average of external credit scores
df_encoded['EXT_SOURCES_MEAN'] = df_encoded[['EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3']].mean(axis=1)

# ── Fill remaining NaN with median ──
df_encoded = df_encoded.fillna(df_encoded.median(numeric_only=True))

print('Final shape:', df_encoded.shape)

## Step 5: Train-Test Split + SMOTE

In [ ]:
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE

X = df_encoded.drop(['TARGET', 'SK_ID_CURR'], axis=1, errors='ignore')
y = df_encoded['TARGET']

print('Features:', X.shape[1])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Train: {X_train.shape}  Test: {X_test.shape}')
print(f'Train default rate: {y_train.mean():.3f}')

# SMOTE to balance classes
print('\nApplying SMOTE...')
X_res, y_res = SMOTE(random_state=42, k_neighbors=5).fit_resample(X_train, y_train)
print(f'After SMOTE: {y_res.value_counts().to_dict()}')

## Step 6: Train XGBoost Model

In [ ]:
from xgboost import XGBClassifier

model = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=10,   # Handles imbalance
    use_label_encoder=False,
    eval_metric='auc',
    random_state=42,
    n_jobs=-1,
)

model.fit(
    X_res, y_res,
    eval_set=[(X_test, y_test)],
    verbose=50
)

print('Training complete!')

## Step 7: Evaluate Model

In [ ]:
from sklearn.metrics import (
    roc_auc_score, classification_report,
    confusion_matrix, RocCurveDisplay
)
import matplotlib.pyplot as plt

y_pred       = model.predict(X_test)
y_pred_proba = model.predict_proba(X_test)[:, 1]

auc = roc_auc_score(y_test, y_pred_proba)
print(f'AUC-ROC: {auc:.4f}')
print()
print(classification_report(y_test, y_pred, target_names=['No Default', 'Default']))

# ROC Curve
RocCurveDisplay.from_predictions(y_test, y_pred_proba)
plt.title(f'ROC Curve — AUC: {auc:.3f}')
plt.savefig('../docs/roc_curve.png', dpi=120, bbox_inches='tight')
plt.show()

## Step 8: SHAP Explainability

In [ ]:
import shap

print('Building SHAP explainer...')
explainer = shap.TreeExplainer(model)

# Use a sample for speed
sample = X_test.iloc[:500]
shap_values = explainer.shap_values(sample)

print('SHAP values shape:', shap_values.shape)

# ── Global Summary Plot (which features matter most?) ──
plt.figure(figsize=(10, 7))
shap.summary_plot(shap_values, sample, max_display=15, show=False)
plt.title('SHAP Feature Importance — Top 15 Features')
plt.tight_layout()
plt.savefig('../docs/shap_summary.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# ── Waterfall Plot — single applicant explanation ──
idx = 0  # Change this to explain any applicant

shap.waterfall_plot(
    shap.Explanation(
        values=shap_values[idx],
        base_values=explainer.expected_value,
        feature_names=sample.columns.tolist(),
        data=sample.iloc[idx].values
    ),
    max_display=12
)

prob = model.predict_proba(sample.iloc[[idx]])[0][1]
decision = 'APPROVED' if prob < 0.30 else 'MANUAL REVIEW' if prob < 0.55 else 'REJECTED'
print(f'Applicant #{idx}: Default Probability = {prob*100:.1f}% → {decision}')

## Step 9: Save Model

In [ ]:
import pickle
import os

save_path = '../backend/model.pkl'
with open(save_path, 'wb') as f:
    pickle.dump(model, f)

size_mb = os.path.getsize(save_path) / (1024 * 1024)
print(f'Model saved to {save_path}')
print(f'File size: {size_mb:.1f} MB')
print(f'Features: {len(model.feature_names_in_)}')
print(f'AUC-ROC: {auc:.4f}')